# 01 Build the Housing Completion Dataset

This notebook combines the annual Planning London Datahub exports, cleans and deduplicates completed housing records, geocodes valid postcodes with the ONS Postcode Directory, and assigns PTAL and borough attributes.


## 1. Import package

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

## 2. Set file path

In [2]:
from pathlib import Path

PROJECT_ROOT = Path(".")
BASE = PROJECT_ROOT

PLD_FILES = sorted(BASE.glob("New Housing Applications - Completed*.csv"))

BOROUGH_PATH = BASE / "London_Boroughs.gpkg"
PTAL_PATH = BASE / "PTAL_2023_Grid_100mx100m_Data.geojson"
ONSPD_PATH = BASE / "ONSPD_FEB_2026_UK.csv"

print("Number of PLD files:", len(PLD_FILES))
for f in PLD_FILES:
    print(f.name)

print("\nBorough file exists:", BOROUGH_PATH.exists())
print("PTAL file exists:", PTAL_PATH.exists())
print("ONSPD file exists:", ONSPD_PATH.exists())

Number of PLD files: 11
New Housing Applications - Completed 06_04_2015-05_04_2016.csv
New Housing Applications - Completed 06_04_2016-05_04_2017.csv
New Housing Applications - Completed 06_04_2017-05_04_2018.csv
New Housing Applications - Completed 06_04_2018-05_04_2019.csv
New Housing Applications - Completed 06_04_2019-05_04_2020.csv
New Housing Applications - Completed 06_04_2020-05_04_2021.csv
New Housing Applications - Completed 06_04_2021-05_04_2022.csv
New Housing Applications - Completed 06_04_2022-05_04_2023.csv
New Housing Applications - Completed 06_04_2023-05_04_2024.csv
New Housing Applications - Completed 06_04_2024-05_04_2025.csv
New Housing Applications - Completed 06_04_2025-05_04_2026.csv

Borough file exists: True
PTAL file exists: True
ONSPD file exists: True


## 3. Read and merge all PLD completed housing files

In [3]:
from pathlib import Path

# "." 表示 notebook 当前所在的文件夹
PROJECT_ROOT = Path(".")
BASE = PROJECT_ROOT

PLD_FILES = sorted(
    BASE.glob("New Housing Applications - Completed*.csv")
)

BOROUGH_PATH = BASE / "London_Boroughs.gpkg"
PTAL_PATH = BASE / "PTAL_2023_Grid_100mx100m_Data.geojson"
ONSPD_PATH = BASE / "ONSPD_FEB_2026_UK.csv"

print("Working directory:", Path.cwd())
print("Number of PLD files:", len(PLD_FILES))

for f in PLD_FILES:
    print(f.name)

print("\nBorough file exists:", BOROUGH_PATH.exists())
print("PTAL file exists:", PTAL_PATH.exists())
print("ONSPD file exists:", ONSPD_PATH.exists())

if not PLD_FILES:
    raise FileNotFoundError(
        "No PLD CSV files were found in the notebook folder."
    )

Working directory: C:\Users\YOLO\GitHub\CASA\dissertation\project\notebook\code
Number of PLD files: 11
New Housing Applications - Completed 06_04_2015-05_04_2016.csv
New Housing Applications - Completed 06_04_2016-05_04_2017.csv
New Housing Applications - Completed 06_04_2017-05_04_2018.csv
New Housing Applications - Completed 06_04_2018-05_04_2019.csv
New Housing Applications - Completed 06_04_2019-05_04_2020.csv
New Housing Applications - Completed 06_04_2020-05_04_2021.csv
New Housing Applications - Completed 06_04_2021-05_04_2022.csv
New Housing Applications - Completed 06_04_2022-05_04_2023.csv
New Housing Applications - Completed 06_04_2023-05_04_2024.csv
New Housing Applications - Completed 06_04_2024-05_04_2025.csv
New Housing Applications - Completed 06_04_2025-05_04_2026.csv

Borough file exists: True
PTAL file exists: True
ONSPD file exists: True


In [5]:
import pandas as pd
import warnings


def read_pld_file(path):
    """Read one PLD CSV file."""
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        df = pd.read_csv(
            path,
            engine="python",
            on_bad_lines="skip",
            encoding="utf-8-sig"
        )

    df["source_file"] = path.name
    return df


pld_list = []

for f in PLD_FILES:
    try:
        df = read_pld_file(f)
        pld_list.append(df)
        print(f"Read {f.name}: {len(df):,} rows")

    except Exception as e:
        print(f"FAILED {f.name}")
        print(repr(e))


if not pld_list:
    raise RuntimeError(
        "None of the PLD files were successfully read."
    )


pld_raw = pd.concat(
    pld_list,
    ignore_index=True
)

print("\nPLD raw shape:", pld_raw.shape)
pld_raw.head()

Read New Housing Applications - Completed 06_04_2015-05_04_2016.csv: 4,432 rows
Read New Housing Applications - Completed 06_04_2016-05_04_2017.csv: 6,161 rows
Read New Housing Applications - Completed 06_04_2017-05_04_2018.csv: 5,060 rows
Read New Housing Applications - Completed 06_04_2018-05_04_2019.csv: 4,888 rows
Read New Housing Applications - Completed 06_04_2019-05_04_2020.csv: 4,285 rows
Read New Housing Applications - Completed 06_04_2020-05_04_2021.csv: 3,455 rows
Read New Housing Applications - Completed 06_04_2021-05_04_2022.csv: 3,696 rows
Read New Housing Applications - Completed 06_04_2022-05_04_2023.csv: 1,077 rows
Read New Housing Applications - Completed 06_04_2023-05_04_2024.csv: 3,656 rows
Read New Housing Applications - Completed 06_04_2024-05_04_2025.csv: 2,697 rows
Read New Housing Applications - Completed 06_04_2025-05_04_2026.csv: 704 rows

PLD raw shape: (40111, 26)


,LPA Number,Borough,Valid date,Status,Application type,Description,Site name,Site number,Street name,Locality,...,Units Lost,Net Units,Net units with commencement date,Net units with completion date,Decision,Decision date,Appeal decision,Appeal decision date,URL planning application,source_file
0,221164FUL,Ealing,17/03/2022,Completed,All Other,Change of use of single family dwellinghouse (...,65 Twyford Abbey Road,NaN,Twyford Abbey Road,London,...,0.0,14.0,14.0,14.0,Refused,28/03/2023,NaN,NaN,https://pam.ealing.gov.uk/online-applications/...,New Housing Applications - Completed 06_04_201...
1,13/4745/FUL,Richmond upon Thames,19/12/2013,Completed,All Other,One and two storey extensions to existing hous...,NaN,20,Holmesdale Road,NaN,...,0.0,1.0,1.0,1.0,Approved,15/07/2014,NaN,NaN,NaN,New Housing Applications - Completed 06_04_201...
2,H/00978/14,Barnet,NaN,Completed,All Other,Use as 4 no. self-contained flats.,190,190,"Watford Way,","London,",...,3.0,1.0,1.0,1.0,Approved,06/05/2014,NaN,NaN,NaN,New Housing Applications - Completed 06_04_201...
3,PP/2014/5274,Ealing,13/10/2014,Completed,All Other,Conversion of dwellinghouse into three flats (...,87 Gunnersbury Avenue,87,Gunnersbury Avenue,London,...,1.0,2.0,2.0,2.0,Approved,08/12/2014,NaN,NaN,https://pam.ealing.gov.uk/online-applications/...,New Housing Applications - Completed 06_04_201...
4,P110524,Islington,28/04/2011,Completed,All Other,Change of use of basement of existing dwelling...,NaN,183,Southgate Road,London,...,0.0,1.0,1.0,1.0,Approved,23/06/2011,NaN,NaN,NaN,New Housing Applications - Completed 06_04_201...


In [6]:
print("Number of PLD files read:", len(pld_list))
print("Total rows:", len(pld_raw))

pld_raw.columns.tolist()

Number of PLD files read: 11
Total rows: 40111


['LPA Number',
 'Borough',
 'Valid date',
 'Status',
 'Application type',
 'Description',
 'Site name',
 'Site number',
 'Street name',
 'Locality',
 'Postcode',
 'Actual commencement date',
 'Actual completion date',
 'Total number of proposed residential units',
 'Total number of existing residential units',
 'Units Proposed',
 'Units Lost',
 'Net Units',
 'Net units with commencement date',
 'Net units with completion date',
 'Decision',
 'Decision date',
 'Appeal decision',
 'Appeal decision date',
 'URL planning application',
 'source_file']

## 4. Clean PLD data

In [7]:
import re

pld = pld_raw.copy()

pld["completion_date"] = pd.to_datetime(
    pld["Actual completion date"],
    dayfirst=True,
    errors="coerce"
)

pld["valid_date"] = pd.to_datetime(
    pld["Valid date"],
    dayfirst=True,
    errors="coerce"
)

pld["completed_units"] = pd.to_numeric(
    pld["Net units with completion date"],
    errors="coerce"
)

postcode_pattern = r"([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})"

def extract_postcode(value):
    if pd.isna(value):
        return np.nan
    
    text = str(value).upper()
    match = re.search(postcode_pattern, text)
    
    if match:
        return re.sub(r"\s+", "", match.group(1))
    else:
        return np.nan

# First extract postcode from the original Postcode column
pld["postcode_from_postcode_col"] = pld["Postcode"].apply(extract_postcode)

pld["postcode_clean"] = pld["postcode_from_postcode_col"]
pld["postcode_source"] = pd.NA

pld.loc[
    pld["postcode_clean"].notna(),
    "postcode_source"
] = "Postcode"

# Try to recover postcodes from other PLD address-related fields
postcode_candidate_cols = [
    "Site name",
    "Street name",
    "Locality",
    "Description"
]

postcode_candidate_cols = [
    col for col in postcode_candidate_cols
    if col in pld.columns
]

for col in postcode_candidate_cols:
    extracted = pld[col].apply(extract_postcode)
    
    mask = (
        pld["postcode_clean"].isna() &
        extracted.notna()
    )
    
    pld.loc[mask, "postcode_clean"] = extracted[mask]
    pld.loc[mask, "postcode_source"] = col

# Core completed housing dataset before postcode/geocoding exclusions
pld_core = pld[
    (pld["Status"] == "Completed") &
    (pld["completion_date"].notna()) &
    (pld["completed_units"].notna()) &
    (pld["completed_units"] > 0)
].copy()

# Analysis dataset with a valid extractable postcode
pld_clean = pld_core[
    pld_core["postcode_clean"].notna()
].copy()

print("Raw rows:", len(pld_raw))
print("Core completed rows:", len(pld_core))
print("Core completed units:", pld_core["completed_units"].sum())
print("Rows with valid postcode:", len(pld_clean))
print("Units with valid postcode:", pld_clean["completed_units"].sum())
print("Rows without valid postcode:", len(pld_core) - len(pld_clean))
print("Units without valid postcode:", pld_core["completed_units"].sum() - pld_clean["completed_units"].sum())

Raw rows: 40111
Core completed rows: 31384
Core completed units: 392548.0
Rows with valid postcode: 29190
Units with valid postcode: 291507.0
Rows without valid postcode: 2194
Units without valid postcode: 101041.0


In [8]:
print("Postcode source among core completed records:")
print(pld_core["postcode_source"].value_counts(dropna=False))

rescued_mask = (
    pld_core["postcode_from_postcode_col"].isna() &
    pld_core["postcode_clean"].notna()
)

print("\nExtra records rescued from non-Postcode columns:", rescued_mask.sum())
print(
    "Extra units rescued from non-Postcode columns:",
    pld_core.loc[rescued_mask, "completed_units"].sum()
)

Postcode source among core completed records:
postcode_source
Postcode       27444
<NA>            2194
Site name       1735
Description       11
Name: count, dtype: int64

Extra records rescued from non-Postcode columns: 1746
Extra units rescued from non-Postcode columns: 11075.0


In [9]:
from pathlib import Path

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

no_postcode_check_cols = [
    "LPA Number",
    "Borough",
    "Postcode",
    "postcode_clean",
    "postcode_source",
    "completed_units",
    "completion_date",
    "Site name",
    "Street name",
    "Locality",
    "Description",
    "source_file"
]

no_postcode_check_cols = [
    col for col in no_postcode_check_cols
    if col in pld_core.columns
]

no_postcode_check = (
    pld_core[pld_core["postcode_clean"].isna()]
    [no_postcode_check_cols]
    .sort_values("completed_units", ascending=False)
)

display(no_postcode_check.head(50))

no_postcode_check.to_csv(
    OUTPUT_DIR / "quality_check_no_extractable_postcode_top_records.csv",
    index=False
)

,LPA Number,Borough,Postcode,postcode_clean,postcode_source,completed_units,completion_date,Site name,Street name,Locality,Description,source_file
9820,11/2366/O,Greenwich,SE3,NaN,<NA>,1260.0,2016-11-10,"Kidbrooke Village, Phase 4",Kidbrooke Park Road,NaN,Demolition of existing buildings and erection ...,New Housing Applications - Completed 06_04_201...
35789,H/01054/13,London Borough of Barnet,NaN,NaN,<NA>,1174.0,2023-10-10,"West Hendon Estate, West Hendon, London, NW9",NaN,NaN,Hybrid planning application for the demolition...,New Housing Applications - Completed 06_04_202...
24260,16/0186/MA,Greenwich,SE10,NaN,<NA>,1007.0,2020-02-21,"Plot N0205, N0206 and N0207",Peninsula Square,Greenwich,Demolition of building on Plot N0205 (the Rotu...,New Housing Applications - Completed 06_04_201...
35683,14/00703/OUT,Barking and Dagenham,NaN,NaN,<NA>,968.0,2024-03-25,Gascoigne Estate East,KING EDWARDS ROAD,NaN,Hybrid (part full/part outline) application fo...,New Housing Applications - Completed 06_04_202...
24184,14/00703/OUT,Barking and Dagenham,NaN,NaN,<NA>,968.0,2024-03-25,Gascoigne Estate East,KING EDWARDS ROAD,NaN,Hybrid (part full/part outline) application fo...,New Housing Applications - Completed 06_04_201...
9994,04/01230/OUT,London Borough of Barking and Dagenham,NaN,NaN,<NA>,818.0,2017-03-30,Barking Riverside Area,RENWICK ROAD,Barking,Development comprising or to provide a mixed u...,New Housing Applications - Completed 06_04_201...
1173,DC09/71246,Lewisham,SE13,NaN,<NA>,794.0,2015-12-10,LAND ON SOUTH SIDE,LOAMPIT VALE,NaN,The construction of eight buildings ranging fr...,New Housing Applications - Completed 06_04_201...
31227,PA/18/01525,Tower Hamlets,E14,NaN,<NA>,767.0,2022-03-14,Arrowhead Quay East,Of 163 Marsh Wall,NaN,Application for variation of condition no. 1 (...,New Housing Applications - Completed 06_04_202...
1881,10/02175/P,Croydon,CR0,NaN,<NA>,755.0,2015-12-07,Saffron Square,Wellesley Road,Croydon,Redevelopment of former Randolph and Pembroke ...,New Housing Applications - Completed 06_04_201...
27797,14/2607/F,Greenwich,SE3,NaN,<NA>,751.0,2021-04-01,Kidbrooke Village Phase 3,Kidbrooke Park Road,NaN,Demolition of existing buildings and construct...,New Housing Applications - Completed 06_04_202...


In [10]:
# Check potential duplicate completion records

duplicate_key_cols = [
    "LPA Number",
    "Borough",
    "Postcode",
    "completed_units",
    "completion_date",
    "Site name",
    "Street name"
]

duplicate_key_cols = [
    col for col in duplicate_key_cols
    if col in pld_core.columns
]

potential_duplicates = pld_core[
    pld_core.duplicated(subset=duplicate_key_cols, keep=False)
].copy()

potential_duplicates = potential_duplicates.sort_values(
    duplicate_key_cols
)

print("Potential duplicate rows:", len(potential_duplicates))
print("Potential duplicate completed units:", potential_duplicates["completed_units"].sum())

display(
    potential_duplicates[
        duplicate_key_cols + ["Description", "source_file"]
    ].head(100)
)

Potential duplicate rows: 5890
Potential duplicate completed units: 159758.0


,LPA Number,Borough,Postcode,completed_units,completion_date,Site name,Street name,Description,source_file
14654,0039/17,Redbridge,NaN,1.0,2018-09-27,49,York Mews,Conversion of upper floors to provide 2 no. se...,New Housing Applications - Completed 06_04_201...
19565,0039/17,Redbridge,NaN,1.0,2018-09-27,49,York Mews,Conversion of upper floors to provide 2 no. se...,New Housing Applications - Completed 06_04_201...
37509,0082/24,Redbridge,NaN,3.0,2025-08-27,Development At The Mews 26a,Roding Lane South,Change of use from Office to 3no. self contain...,New Housing Applications - Completed 06_04_202...
39494,0082/24,Redbridge,NaN,3.0,2025-08-27,Development At The Mews 26a,Roding Lane South,Change of use from Office to 3no. self contain...,New Housing Applications - Completed 06_04_202...
8095,0577/13,Redbridge,IG4 5HN,1.0,2019-10-08,23,Grangeway Gardens,(AMENDED DESCRIPTION) Conversion of existing d...,New Housing Applications - Completed 06_04_201...
...,...,...,...,...,...,...,...,...,...
3443,12/08720/FULL,Westminster,W1H 4NN,1.0,2017-04-30,42,Homer Street,"Use of first, second and third floors as 1x1 a...",New Housing Applications - Completed 06_04_201...
13736,12/08720/FULL,Westminster,W1H 4NN,1.0,2017-04-30,42,Homer Street,"Use of first, second and third floors as 1x1 a...",New Housing Applications - Completed 06_04_201...
7839,12/12732/FUL,Kingston upon Thames,KT2 6LJ,1.0,2017-10-26,90-92,Willoughby Road,Conversion of existing retail unit on ground f...,New Housing Applications - Completed 06_04_201...
13217,12/12732/FUL,Kingston upon Thames,KT2 6LJ,1.0,2017-10-26,90-92,Willoughby Road,Conversion of existing retail unit on ground f...,New Housing Applications - Completed 06_04_201...


In [11]:
duplicate_summary = (
    potential_duplicates
    .groupby(duplicate_key_cols, dropna=False)
    .agg(
        duplicate_rows=("completed_units", "size"),
        duplicated_units_total=("completed_units", "sum"),
        one_record_units=("completed_units", "first"),
        source_files=("source_file", lambda x: " | ".join(sorted(x.astype(str).unique())))
    )
    .reset_index()
    .sort_values("duplicated_units_total", ascending=False)
)

display(duplicate_summary.head(50))

duplicate_summary.to_csv(
    OUTPUT_DIR / "quality_check_potential_duplicate_records.csv",
    index=False
)

,LPA Number,Borough,Postcode,completed_units,completion_date,Site name,Street name,duplicate_rows,duplicated_units_total,one_record_units,source_files
1518,2014/04726/OUT,Hammersmith & Fulham,W12 7RQ,1465.0,2023-09-06,54,Wood Lane,3,4395.0,1465.0,New Housing Applications - Completed 06_04_202...
153,14/02893/FUL,Newham,E13 9AZ,842.0,2023-06-01,West Ham United Football Club,Green Street,4,3368.0,842.0,New Housing Applications - Completed 06_04_201...
2784,PA/16/02336,Tower Hamlets,E14 9SL,907.0,2021-09-04,"Enterprise Business Park, 2",Millharbour,3,2721.0,907.0,New Housing Applications - Completed 06_04_201...
93,13/3025,Greenwich,SE10,661.0,2018-05-25,Land at,Enderby Wharf,4,2644.0,661.0,New Housing Applications - Completed 06_04_201...
1462,2011/3748,Wandsworth,SW8 5BP,813.0,2018-03-31,"Tideway Industrial Estate, 87",Kirtling Street,3,2439.0,813.0,New Housing Applications - Completed 06_04_201...
908,17/0076/FUMOPDC,Brent,NW10 7HQ,807.0,2023-03-31,Land at First Central site,Lakeside Drive,3,2421.0,807.0,New Housing Applications - Completed 06_04_202...
4,0734/01M,Redbridge,IG7,398.0,2022-06-01,Land at,Five Oaks Lane,6,2388.0,398.0,New Housing Applications - Completed 06_04_201...
2658,P0940.18,Havering,RM12 6RS,776.0,2022-04-14,St George's Hospital Phase 1,Suttons Lane,3,2328.0,776.0,New Housing Applications - Completed 06_04_201...
1457,2011/00407/COMB,Hammersmith & Fulham,W6,570.0,2023-04-27,Hammersmith Embankment,Chancellor's Road,4,2280.0,570.0,New Housing Applications - Completed 06_04_201...
2497,P/2008/0156,Ealing,W5 2XA,680.0,2018-07-01,LAND AT DICKENS YARD & CHURCH 2-12,NEW BROADWAY,3,2040.0,680.0,New Housing Applications - Completed 06_04_201...


In [12]:
dedup_key_cols = [
    "LPA Number",
    "Borough",
    "Postcode",
    "completed_units",
    "completion_date",
    "Site name",
    "Street name"
]

dedup_key_cols = [
    col for col in dedup_key_cols
    if col in pld_core.columns
]

pld_core_dedup = (
    pld_core
    .sort_values("source_file")
    .drop_duplicates(subset=dedup_key_cols, keep="first")
    .copy()
)

print("Before dedup rows:", len(pld_core))
print("After dedup rows:", len(pld_core_dedup))
print("Removed rows:", len(pld_core) - len(pld_core_dedup))

print("Before dedup units:", pld_core["completed_units"].sum())
print("After dedup units:", pld_core_dedup["completed_units"].sum())
print("Removed duplicated units:", pld_core["completed_units"].sum() - pld_core_dedup["completed_units"].sum())

Before dedup rows: 31384
After dedup rows: 28395
Removed rows: 2989
Before dedup units: 392548.0
After dedup units: 300933.0
Removed duplicated units: 91615.0


In [13]:
print("Rows with valid postcode:", pld_core_dedup["postcode_clean"].notna().sum())
print(
    "Units with valid postcode:",
    pld_core_dedup.loc[pld_core_dedup["postcode_clean"].notna(), "completed_units"].sum()
)

print("Rows without valid postcode:", pld_core_dedup["postcode_clean"].isna().sum())
print(
    "Units without valid postcode:",
    pld_core_dedup.loc[pld_core_dedup["postcode_clean"].isna(), "completed_units"].sum()
)

Rows with valid postcode: 26375
Units with valid postcode: 229669.0
Rows without valid postcode: 2020
Units without valid postcode: 71264.0


## 5. Recalculate the financial year using the actual completion date

In [14]:
def financial_year_from_date(date):
    if pd.isna(date):
        return np.nan
    
    year = date.year
    
    if date >= pd.Timestamp(year=year, month=4, day=6):
        start_year = year
    else:
        start_year = year - 1
    
    return f"{start_year}/{str(start_year + 1)[-2:]}"
    
pld_core["completion_financial_year"] = pld_core["completion_date"].apply(financial_year_from_date)
pld_core["completion_calendar_year"] = pld_core["completion_date"].dt.year

pld_clean["completion_financial_year"] = pld_clean["completion_date"].apply(financial_year_from_date)
pld_clean["completion_calendar_year"] = pld_clean["completion_date"].dt.year

annual_summary = (
    pld_core
    .groupby("completion_financial_year")
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
    .sort_values("completion_financial_year")
)

annual_summary

,completion_financial_year,records,completed_units
0,2015/16,2955,25475.0
1,2016/17,4614,47299.0
2,2017/18,4051,46616.0
3,2018/19,4079,47028.0
4,2019/20,4100,47838.0
5,2020/21,2811,36671.0
6,2021/22,2855,38713.0
7,2022/23,928,23728.0
8,2023/24,2424,41493.0
9,2024/25,1996,32107.0


In [15]:
# Define the final main analysis dataset:
# deduplicated PLD records, restricted to 2015/16–2024/25

main_years = [
    "2015/16", "2016/17", "2017/18", "2018/19", "2019/20",
    "2020/21", "2021/22", "2022/23", "2023/24", "2024/25"
]

# Add financial year fields to the deduplicated dataset
pld_core_dedup["completion_financial_year"] = (
    pld_core_dedup["completion_date"].apply(financial_year_from_date)
)

pld_core_dedup["completion_calendar_year"] = (
    pld_core_dedup["completion_date"].dt.year
)

# Keep a full deduplicated copy for audit if needed
pld_core_all_years = pld_core_dedup.copy()

# Main analysis dataset only
pld_core_main = pld_core_dedup[
    pld_core_dedup["completion_financial_year"].isin(main_years)
].copy()

# From this point onwards, use the main analysis dataset
pld_core = pld_core_main.copy()

pld_clean = pld_core[
    pld_core["postcode_clean"].notna()
].copy()

print("Final main analysis rows:", len(pld_core))
print("Final main analysis units:", pld_core["completed_units"].sum())

print("Rows with valid postcode:", len(pld_clean))
print("Units with valid postcode:", pld_clean["completed_units"].sum())

print("Rows without valid postcode:", len(pld_core) - len(pld_clean))
print("Units without valid postcode:", pld_core["completed_units"].sum() - pld_clean["completed_units"].sum())

Final main analysis rows: 27863
Final main analysis units: 296746.0
Rows with valid postcode: 25922
Units with valid postcode: 226657.0
Rows without valid postcode: 1941
Units without valid postcode: 70089.0


In [16]:
annual_summary = (
    pld_core
    .groupby("completion_financial_year")
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
    .sort_values("completion_financial_year")
)

annual_summary.to_csv(
    OUTPUT_DIR / "annual_pld_completion_summary.csv",
    index=False
)

annual_summary

,completion_financial_year,records,completed_units
0,2015/16,2955,25475.0
1,2016/17,4191,41183.0
2,2017/18,3445,33885.0
3,2018/19,3423,34050.0
4,2019/20,3364,35953.0
5,2020/21,2630,28463.0
6,2021/22,2781,30606.0
7,2022/23,887,13663.0
8,2023/24,2363,29116.0
9,2024/25,1824,24352.0


## 6. Read the ONS Postcode Directory and retain only the required fields

In [17]:
raw_onspd_columns = pd.read_csv(
    ONSPD_PATH,
    nrows=0,
    encoding="utf-8-sig"
).columns.tolist()

print("First 80 ONSPD columns:")
print(raw_onspd_columns[:80])

# Your ONSPD version uses east1m / north1m
usecols = ["pcds", "east1m", "north1m", "doterm"]

if "lsoa11cd" in raw_onspd_columns:
    usecols.append("lsoa11cd")

if "lsoa21cd" in raw_onspd_columns:
    usecols.append("lsoa21cd")

print("Using columns:", usecols)

onspd = pd.read_csv(
    ONSPD_PATH,
    usecols=usecols,
    dtype=str,
    encoding="utf-8-sig"
)

onspd.head()

First 80 ONSPD columns:
['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'cty25cd', 'ced25cd', 'lad25cd', 'wd25cd', 'parncp25cd', 'usrtypind', 'east1m', 'north1m', 'gridind', 'hlth19cd', 'nhser24cd', 'ctry25cd', 'rgn25cd', 'ssr95cd', 'pcon24cd', 'eer20cd', 'educ23cd', 'ttwa15cd', 'pco19cd', 'itl25cd', 'wdstl05cd', 'oa01cd', 'wdcas03cd', 'npark16cd', 'lsoa01cd', 'msoa01cd', 'ruc01ind', 'oac01ind', 'oa11cd', 'lsoa11cd', 'msoa11cd', 'wz11cd', 'sicbl24cd', 'bua24cd', 'ruc11ind', 'oac11ind', 'lat', 'long', 'lep21cd1', 'lep21cd2', 'pfa23cd', 'imd20ind', 'cal24cd', 'icb23cd', 'oa21cd', 'lsoa21cd', 'msoa21cd', 'ruc21ind']
Using columns: ['pcds', 'east1m', 'north1m', 'doterm', 'lsoa11cd', 'lsoa21cd']


,pcds,doterm,east1m,north1m,lsoa11cd,lsoa21cd
0,AB1 0AA,199606,385386,0801193,S01006514,S01013490
1,AB1 0AB,199606,385177,0801314,S01006514,S01013490
2,AB1 0AD,199606,385053,0801092,S01006514,S01013490
3,AB1 0AE,199606,384600,0799300,S01006853,S01013856
4,AB1 0AF,199207,384460,0800660,S01006511,S01013487


In [18]:
print("ONSPD LSOA columns:")
print([c for c in onspd.columns if "lsoa" in c.lower()])

ONSPD LSOA columns:
['lsoa11cd', 'lsoa21cd']


In [19]:
# Standardise column names
rename_dict = {
    "pcds": "pcds",
    "east1m": "oseast1m",
    "north1m": "osnrth1m",
    "doterm": "doterm"
}

if "lsoa21cd" in onspd.columns:
    rename_dict["lsoa21cd"] = "lsoa21cd"

onspd = onspd.rename(columns=rename_dict)

# Clean postcode
onspd["postcode_clean"] = (
    onspd["pcds"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)

# Convert coordinates to numbers
onspd["oseast1m"] = pd.to_numeric(onspd["oseast1m"], errors="coerce")
onspd["osnrth1m"] = pd.to_numeric(onspd["osnrth1m"], errors="coerce")

# Keep records with valid postcode and coordinates
onspd = onspd[
    onspd["postcode_clean"].notna() &
    onspd["oseast1m"].notna() &
    onspd["osnrth1m"].notna()
].copy()

print("ONSPD rows after cleaning:", len(onspd))
print(onspd.columns.tolist())
onspd.head()

ONSPD rows after cleaning: 2699393
['pcds', 'doterm', 'oseast1m', 'osnrth1m', 'lsoa11cd', 'lsoa21cd', 'postcode_clean']


,pcds,doterm,oseast1m,osnrth1m,lsoa11cd,lsoa21cd,postcode_clean
0,AB1 0AA,199606,385386.0,801193.0,S01006514,S01013490,AB10AA
1,AB1 0AB,199606,385177.0,801314.0,S01006514,S01013490,AB10AB
2,AB1 0AD,199606,385053.0,801092.0,S01006514,S01013490,AB10AD
3,AB1 0AE,199606,384600.0,799300.0,S01006853,S01013856,AB10AE
4,AB1 0AF,199207,384460.0,800660.0,S01006511,S01013487,AB10AF


## 7. Match PLD and ONSPD, and add coordinates to housing completions

In [20]:
pld_geocode = pld_clean.merge(
    onspd,
    on="postcode_clean",
    how="left",
    indicator=True
)

In [21]:
print("PLD geocode LSOA columns:")
print([c for c in pld_geocode.columns if "lsoa" in c.lower()])

PLD geocode LSOA columns:
['lsoa11cd', 'lsoa21cd']


In [22]:
print("Records with valid postcode:", len(pld_clean))
print("Units with valid postcode:", pld_clean["completed_units"].sum())

print("Matched records:", (pld_geocode["_merge"] == "both").sum())
print("Unmatched records:", (pld_geocode["_merge"] == "left_only").sum())

print(
    "Matched units:",
    pld_geocode.loc[
        pld_geocode["_merge"] == "both",
        "completed_units"
    ].sum()
)

print(
    "Unmatched units:",
    pld_geocode.loc[
        pld_geocode["_merge"] == "left_only",
        "completed_units"
    ].sum()
)

print(
    "Unit geocoding coverage from valid-postcode records:",
    pld_geocode.loc[
        pld_geocode["_merge"] == "both",
        "completed_units"
    ].sum() / pld_clean["completed_units"].sum()
)

print(
    "Unit geocoding coverage from all completed records:",
    pld_geocode.loc[
        pld_geocode["_merge"] == "both",
        "completed_units"
    ].sum() / pld_core["completed_units"].sum()
)

Records with valid postcode: 25922
Units with valid postcode: 226657.0
Matched records: 25852
Unmatched records: 70
Matched units: 225451.0
Unmatched units: 1206.0
Unit geocoding coverage from valid-postcode records: 0.9946791848475891
Unit geocoding coverage from all completed records: 0.7597440235083203


In [23]:
unmatched = pld_geocode[
    pld_geocode["_merge"] == "left_only"
].copy()

unmatched_check = unmatched[
    [
        "LPA Number",
        "Borough",
        "Postcode",
        "postcode_clean",
        "completed_units",
        "completion_financial_year",
        "Site name",
        "Street name",
        "Locality",
        "Description",
        "source_file"
    ]
].sort_values("completed_units", ascending=False)

unmatched_check.head(30)

,LPA Number,Borough,Postcode,postcode_clean,completed_units,completion_financial_year,Site name,Street name,Locality,Description,source_file
24860,DC/19/111861,Lewisham,NaN,SE107QR,443.0,2024/25,"HEATHSIDE AND LETHBRIDGE (PHASE 5 and 6), BLAC...",NaN,NaN,Application submitted for the approval of rese...,New Housing Applications - Completed 06_04_202...
6870,2011/1975,Hackney,E8 2TJ,E82TJ,394.0,2016/17,Haggerstone West and Kingsland Estates Phase 2,Kingsland Road,NaN,The submission of details of: The Reserved Mat...,New Housing Applications - Completed 06_04_201...
6829,2013/0405/P,Camden,N1C 4WH,N1C4WH,129.0,2016/17,"SA1 (Allocated Site), Building T1, Kings Cross...",York Way,NaN,Reserved matters in connection with the erecti...,New Housing Applications - Completed 06_04_201...
24676,2019/02351/FUL,London Borough of Hammersmith and Fulham,NaN,NW149EF,36.0,2024/25,Ada Lewis House 2 Palliser RoadLondonW14 9EF,NaN,NaN,Demolition of the existing building and the er...,New Housing Applications - Completed 06_04_202...
24677,DC/18/107715,Lewisham,SE10 7QR,SE107QR,34.0,2024/25,Heathside and Lethbridge Estates phase 5 & 6,Blackheath Hill,NaN,Application submitted under Section 73 of the ...,New Housing Applications - Completed 06_04_202...
23168,20/1360/FUL,London Borough of Barnet,NaN,NN129RW,24.0,2023/24,912 - 920 High RoadLondonN12 9RW,NaN,NaN,Retention of ground floor level and demolition...,New Housing Applications - Completed 06_04_202...
18929,17/02924,Newham,E12 1PG,E121PG,9.0,2020/21,3-5,Windmill Lane,Stratford,Demolition of various structures to the rear o...,New Housing Applications - Completed 06_04_202...
18239,19/03202/PRECUJ,London Borough of Newham,NaN,NE154JF,8.0,2020/21,187 Romford RoadStratfordLondonE15 4JF,NaN,NaN,Prior approval for change of use of office (Us...,New Housing Applications - Completed 06_04_202...
22233,19/02812/FUL,London Borough of Newham,NaN,NE164HR,7.0,2022/23,101-103 Hermit RoadCanning TownLondonE16 4HR,NaN,NaN,Demolition of existing building and constructi...,New Housing Applications - Completed 06_04_202...
16786,14/02425/FUL,Bexley,DA14 5LW,DA145LW,7.0,2020/21,Bedensfield Clinic,Ellenborough Road,Sidcup,Erection of a block of 7 x 3 bedroom terrace t...,New Housing Applications - Completed 06_04_202...


In [24]:
valid_postcode_by_year = (
    pld_clean
    .groupby("completion_financial_year")
    .agg(
        valid_postcode_records=("LPA Number", "count"),
        valid_postcode_units=("completed_units", "sum")
    )
    .reset_index()
)

matched_by_year = (
    pld_geocode[pld_geocode["_merge"] == "both"]
    .groupby("completion_financial_year")
    .agg(
        matched_records=("LPA Number", "count"),
        matched_units=("completed_units", "sum")
    )
    .reset_index()
)

unmatched_by_year = (
    pld_geocode[pld_geocode["_merge"] == "left_only"]
    .groupby("completion_financial_year")
    .agg(
        unmatched_records=("LPA Number", "count"),
        unmatched_units=("completed_units", "sum")
    )
    .reset_index()
)

coverage_by_year = (
    annual_summary
    .merge(valid_postcode_by_year, on="completion_financial_year", how="left")
    .merge(matched_by_year, on="completion_financial_year", how="left")
    .merge(unmatched_by_year, on="completion_financial_year", how="left")
)

cols_to_fill = [
    "valid_postcode_records",
    "valid_postcode_units",
    "matched_records",
    "matched_units",
    "unmatched_records",
    "unmatched_units"
]

coverage_by_year[cols_to_fill] = coverage_by_year[cols_to_fill].fillna(0)

coverage_by_year["units_without_valid_postcode"] = (
    coverage_by_year["completed_units"] -
    coverage_by_year["valid_postcode_units"]
)

coverage_by_year["coverage_from_all_units"] = (
    coverage_by_year["matched_units"] /
    coverage_by_year["completed_units"]
)

coverage_by_year["coverage_from_valid_postcode_units"] = (
    coverage_by_year["matched_units"] /
    coverage_by_year["valid_postcode_units"]
)

coverage_by_year

,completion_financial_year,records,completed_units,valid_postcode_records,valid_postcode_units,matched_records,matched_units,unmatched_records,unmatched_units,units_without_valid_postcode,coverage_from_all_units,coverage_from_valid_postcode_units
0,2015/16,2955,25475.0,2757,19472.0,2746,19451.0,11,21.0,6003.0,0.763533,0.998922
1,2016/17,4191,41183.0,3947,32149.0,3940,31612.0,7,537.0,9034.0,0.767598,0.983297
2,2017/18,3445,33885.0,3297,26938.0,3288,26926.0,9,12.0,6947.0,0.794629,0.999555
3,2018/19,3423,34050.0,3255,26292.0,3249,26280.0,6,12.0,7758.0,0.771806,0.999544
4,2019/20,3364,35953.0,3184,27644.0,3179,27636.0,5,8.0,8309.0,0.768670,0.999711
5,2020/21,2630,28463.0,2375,20764.0,2368,20728.0,7,36.0,7699.0,0.728244,0.998266
6,2021/22,2781,30606.0,2503,21974.0,2494,21957.0,9,17.0,8632.0,0.717408,0.999226
7,2022/23,887,13663.0,815,11083.0,813,11074.0,2,9.0,2580.0,0.810510,0.999188
8,2023/24,2363,29116.0,2129,21512.0,2121,21476.0,8,36.0,7604.0,0.737601,0.998327
9,2024/25,1824,24352.0,1660,18829.0,1654,18311.0,6,518.0,5523.0,0.751930,0.972489


### 7.1 geocoding coverage summary

In [25]:
def pct(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator

core_records = len(pld_core)
core_units = pld_core["completed_units"].sum()

valid_postcode_records = len(pld_clean)
valid_postcode_units = pld_clean["completed_units"].sum()

matched_records = (pld_geocode["_merge"] == "both").sum()
matched_units = pld_geocode.loc[
    pld_geocode["_merge"] == "both",
    "completed_units"
].sum()

unmatched_records = (pld_geocode["_merge"] == "left_only").sum()
unmatched_units = pld_geocode.loc[
    pld_geocode["_merge"] == "left_only",
    "completed_units"
].sum()

no_postcode_records = core_records - valid_postcode_records
no_postcode_units = core_units - valid_postcode_units

geocoding_summary = pd.DataFrame({
    "stage": [
        "Core completed PLD records",
        "Records with extractable postcode",
        "Records without extractable postcode",
        "Matched to ONSPD coordinates",
        "Valid postcode but not matched to ONSPD"
    ],
    "records": [
        core_records,
        valid_postcode_records,
        no_postcode_records,
        matched_records,
        unmatched_records
    ],
    "completed_units": [
        core_units,
        valid_postcode_units,
        no_postcode_units,
        matched_units,
        unmatched_units
    ],
    "record_share_of_core": [
        pct(core_records, core_records),
        pct(valid_postcode_records, core_records),
        pct(no_postcode_records, core_records),
        pct(matched_records, core_records),
        pct(unmatched_records, core_records)
    ],
    "unit_share_of_core": [
        pct(core_units, core_units),
        pct(valid_postcode_units, core_units),
        pct(no_postcode_units, core_units),
        pct(matched_units, core_units),
        pct(unmatched_units, core_units)
    ]
})

geocoding_summary["record_share_of_core"] = geocoding_summary["record_share_of_core"].map(lambda x: f"{x:.2%}")
geocoding_summary["unit_share_of_core"] = geocoding_summary["unit_share_of_core"].map(lambda x: f"{x:.2%}")

geocoding_summary

,stage,records,completed_units,record_share_of_core,unit_share_of_core
0,Core completed PLD records,27863,296746.0,100.00%,100.00%
1,Records with extractable postcode,25922,226657.0,93.03%,76.38%
2,Records without extractable postcode,1941,70089.0,6.97%,23.62%
3,Matched to ONSPD coordinates,25852,225451.0,92.78%,75.97%
4,Valid postcode but not matched to ONSPD,70,1206.0,0.25%,0.41%


In [26]:
geocoding_summary.to_csv(
    OUTPUT_DIR / "summary_geocoding_coverage.csv",
    index=False
)

In [27]:
print(
    f"ONSPD match rate among records with extractable postcodes: "
    f"{matched_records / valid_postcode_records:.2%} of records, "
    f"{matched_units / valid_postcode_units:.2%} of completed units."
)

print(
    f"Final geocoded coverage relative to all core completed PLD data: "
    f"{matched_records / core_records:.2%} of records, "
    f"{matched_units / core_units:.2%} of completed units."
)

print(
    f"Records without extractable postcode account for "
    f"{no_postcode_units / core_units:.2%} of completed units."
)

ONSPD match rate among records with extractable postcodes: 99.73% of records, 99.47% of completed units.
Final geocoded coverage relative to all core completed PLD data: 92.78% of records, 75.97% of completed units.
Records without extractable postcode account for 23.62% of completed units.


## 8. Convert PLD completions into spatial points

In [28]:
pld_points_df = pld_geocode[
    (pld_geocode["_merge"] == "both") &
    (pld_geocode["oseast1m"].notna()) &
    (pld_geocode["osnrth1m"].notna())
].copy()

pld_points = gpd.GeoDataFrame(
    pld_points_df,
    geometry=gpd.points_from_xy(
        pld_points_df["oseast1m"],
        pld_points_df["osnrth1m"]
    ),
    crs="EPSG:27700"
)

print(pld_points.shape)
pld_points.head()

(25852, 42)


,LPA Number,Borough,Valid date,Status,Application type,Description,Site name,Site number,Street name,Locality,...,completion_financial_year,completion_calendar_year,pcds,doterm,oseast1m,osnrth1m,lsoa11cd,lsoa21cd,_merge,geometry
0,221164FUL,Ealing,17/03/2022,Completed,All Other,Change of use of single family dwellinghouse (...,65 Twyford Abbey Road,NaN,Twyford Abbey Road,London,...,2015/16,2016,NW10 7ET,NaN,518888.0,182894.0,E01001276,E01001276,both,POINT (518888 182894)
1,10/03970/FULL,Westminster,NaN,Completed,All Other,Renewal of extant planning permission dated 02...,"Windsor House, 55-56","Windsor House, 55-56",St James's Street,NaN,...,2015/16,2016,SW1A 1LA,NaN,529167.0,180313.0,E01004736,E01004736,both,POINT (529167 180313)
2,11/01869/RG4,Lambeth,NaN,Completed,All Other,Redevelopment of the site involving demolition...,Claremont East Housing Estate,Claremont East Housing Estate,Streatham Hill,NaN,...,2015/16,2015,SW2 3DH,NaN,531483.0,172994.0,E01003174,E01003174,both,POINT (531483 172994)
3,12/AP/3161,Southwark,NaN,Completed,All Other,Conversion of top floor maisonette into two se...,15,15,Risborough Street,NaN,...,2015/16,2015,SE1 0HE,NaN,531974.0,179986.0,E01003927,E01033867,both,POINT (531974 179986)
4,14/10040/FUL,Kingston upon Thames,NaN,Completed,All Other,Conversion of existing house into 1 x 3 bedroo...,28,28,WOODGATE AVENUE,NaN,...,2015/16,2015,KT9 2RA,NaN,517696.0,164444.0,E01002941,E01002941,both,POINT (517696 164444)


In [29]:
pld_points.to_file(
    OUTPUT_DIR / "pld_completed_housing_points.gpkg",
    layer="pld_points",
    driver="GPKG"
)

## 9. Read the PTAL 2023 grid

In [30]:
ptal = gpd.read_file(PTAL_PATH)

print("PTAL CRS:", ptal.crs)
print("PTAL shape:", ptal.shape)
ptal.head()

PTAL CRS: EPSG:4326
PTAL shape: (159451, 12)


,FID,GridID,BUS,LUL,RAIL,TRAM,AI,PTAL_2023,Shape_Leng,Shape__Area,Shape__Length,geometry
0,1,1501,5.100872,0.0,0.0,0.0,5.100872,2,400.0,10000.0,400.0,"POLYGON ((-0.12193 51.31056, -0.1219 51.31146,..."
1,2,1502,3.610841,0.0,0.0,0.0,3.610841,1b,400.0,10000.0,400.0,"POLYGON ((-0.1205 51.31054, -0.12046 51.31144,..."
2,3,1503,3.198376,0.0,0.0,0.0,3.198376,1b,400.0,10000.0,400.0,"POLYGON ((-0.11906 51.31052, -0.11903 51.31142..."
3,4,1504,2.292348,0.0,0.0,0.0,2.292348,1a,400.0,10000.0,400.0,"POLYGON ((-0.11763 51.31049, -0.11759 51.31139..."
4,5,1505,4.662443,0.0,0.0,0.0,4.662443,1b,400.0,10000.0,400.0,"POLYGON ((-0.1162 51.31047, -0.11616 51.31137,..."


In [31]:
ptal.columns.tolist()

['FID',
 'GridID',
 'BUS',
 'LUL',
 'RAIL',
 'TRAM',
 'AI',
 'PTAL_2023',
 'Shape_Leng',
 'Shape__Area',
 'Shape__Length',
 'geometry']

In [32]:
possible_ptal_cols = [c for c in ptal.columns if "PTAL" in c.upper()]
possible_ptal_cols

['PTAL_2023']

In [33]:
PTAL_COL = "PTAL_2023"

In [34]:
ptal = ptal.to_crs("EPSG:27700")

## 10. Match the housing points to the PTAL grid

In [35]:
pld_ptal = gpd.sjoin(
    pld_points,
    ptal[[PTAL_COL, "geometry"]],
    how="left",
    predicate="within"
)

print("Rows after PTAL join:", pld_ptal.shape)
print("Missing PTAL records:", pld_ptal[PTAL_COL].isna().sum())
print("Missing PTAL units:", pld_ptal.loc[pld_ptal[PTAL_COL].isna(), "completed_units"].sum())

pld_ptal[[PTAL_COL, "completed_units", "completion_financial_year"]].head()

Rows after PTAL join: (25852, 44)
Missing PTAL records: 23
Missing PTAL units: 347.0


,PTAL_2023,completed_units,completion_financial_year
0,4,14.0,2015/16
1,6b,3.0,2015/16
2,4,35.0,2015/16
3,6b,1.0,2015/16
4,1b,1.0,2015/16


### 10.1 PTAL join coverage summary

In [36]:
ptal_matched_records = pld_ptal[PTAL_COL].notna().sum()
ptal_missing_records = pld_ptal[PTAL_COL].isna().sum()

ptal_matched_units = pld_ptal.loc[
    pld_ptal[PTAL_COL].notna(),
    "completed_units"
].sum()

ptal_missing_units = pld_ptal.loc[
    pld_ptal[PTAL_COL].isna(),
    "completed_units"
].sum()

ptal_join_summary = pd.DataFrame({
    "stage": [
        "Geocoded housing completion records",
        "Matched to PTAL 2023 grid",
        "Missing PTAL after spatial join"
    ],
    "records": [
        len(pld_ptal),
        ptal_matched_records,
        ptal_missing_records
    ],
    "completed_units": [
        pld_ptal["completed_units"].sum(),
        ptal_matched_units,
        ptal_missing_units
    ],
    "record_share_of_geocoded": [
        "100.00%",
        f"{ptal_matched_records / len(pld_ptal):.2%}",
        f"{ptal_missing_records / len(pld_ptal):.2%}"
    ],
    "unit_share_of_geocoded": [
        "100.00%",
        f"{ptal_matched_units / pld_ptal['completed_units'].sum():.2%}",
        f"{ptal_missing_units / pld_ptal['completed_units'].sum():.2%}"
    ]
})

ptal_join_summary

,stage,records,completed_units,record_share_of_geocoded,unit_share_of_geocoded
0,Geocoded housing completion records,25852,225451.0,100.00%,100.00%
1,Matched to PTAL 2023 grid,25829,225104.0,99.91%,99.85%
2,Missing PTAL after spatial join,23,347.0,0.09%,0.15%


In [37]:
ptal_join_summary.to_csv(
    OUTPUT_DIR / "summary_ptal_join_coverage.csv",
    index=False
)

## 11. Create a PTAL group

Low: 0, 1a, 1b, 2

Medium: 3, 4

High: 5, 6a, 6b

In [38]:
def classify_ptal(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    if value in ["0", "1a", "1b", "2"]:
        return "Low"
    elif value in ["3", "4"]:
        return "Medium"
    elif value in ["5", "6a", "6b"]:
        return "High"
    else:
        return np.nan

pld_ptal["ptal_band"] = pld_ptal[PTAL_COL].apply(classify_ptal)

pld_ptal["ptal_medium_high"] = pld_ptal["ptal_band"].isin(["Medium", "High"])
pld_ptal["ptal_high"] = pld_ptal["ptal_band"].eq("High")

## 12. Read the borough boundary and check the PLD borough field

In [39]:
boroughs = gpd.read_file(BOROUGH_PATH)

print("Borough CRS:", boroughs.crs)
print("Borough shape:", boroughs.shape)
boroughs.head()

Borough CRS: EPSG:27700
Borough shape: (33, 8)


,objectid,name,gss_code,hectares,nonld_area,ons_inner,sub_2011,geometry
0,1,Kingston upon Thames,E09000021,3726.117,0.000,F,South,"POLYGON ((516401.6 160201.8, 516407.3 160210.5..."
1,2,Croydon,E09000008,8649.441,0.000,F,South,"POLYGON ((535009.2 159504.7, 535005.5 159502, ..."
2,3,Bromley,E09000006,15013.487,0.000,F,South,"POLYGON ((540373.6 157530.4, 540361.2 157551.9..."
3,4,Hounslow,E09000018,5658.541,60.755,F,West,"POLYGON ((509703.4 175356.6, 509712.6 175361.8..."
4,5,Ealing,E09000009,5554.428,0.000,F,West,"POLYGON ((515647.2 178787.8, 515608.8 178787.3..."


In [40]:
boroughs.columns.tolist()

['objectid',
 'name',
 'gss_code',
 'hectares',
 'nonld_area',
 'ons_inner',
 'sub_2011',
 'geometry']

In [41]:
possible_borough_cols = [
    c for c in boroughs.columns
    if any(key in c.lower() for key in ["name", "borough", "lad"])
]

possible_borough_cols

['name']

### 12.1 Spatially assign borough names

In [42]:
# Ensure boroughs are in British National Grid
boroughs = boroughs.to_crs("EPSG:27700")

# IMPORTANT: remove old join columns if this cell has been run before
cols_to_drop = [
    "index_right",
    "borough_spatial",
    "name"
]

pld_ptal = pld_ptal.drop(
    columns=[c for c in cols_to_drop if c in pld_ptal.columns],
    errors="ignore"
)

# Prepare borough boundary layer
borough_join = boroughs[["name", "geometry"]].copy()
borough_join = borough_join.rename(columns={"name": "borough_spatial"})

# Spatially assign each housing completion point to a London borough
pld_ptal = gpd.sjoin(
    pld_ptal,
    borough_join,
    how="left",
    predicate="within"
)

# Check missing borough assignments
missing_borough_mask = pld_ptal["borough_spatial"].isna()

print("Missing spatial borough records:", missing_borough_mask.sum())
print(
    "Missing spatial borough units:",
    pld_ptal.loc[missing_borough_mask, "completed_units"].sum()
)

pld_ptal[["Borough", "borough_spatial", "completed_units"]].head(20)

Missing spatial borough records: 23
Missing spatial borough units: 347.0


,Borough,borough_spatial,completed_units
0,Ealing,Ealing,14.0
1,Westminster,Westminster,3.0
2,Lambeth,Lambeth,35.0
3,Southwark,Southwark,1.0
4,Kingston upon Thames,Kingston upon Thames,1.0
5,Bromley,Lewisham,9.0
6,Brent,Brent,1.0
7,Lambeth,Lambeth,2.0
8,Brent,Brent,4.0
9,Lewisham,Lewisham,2.0


In [43]:
# Check whether borough_spatial is now unique
[col for col in pld_ptal.columns if col == "borough_spatial"]

['borough_spatial']

### 12.2 Define main analysis period

In [44]:
main_years = [
    "2015/16",
    "2016/17",
    "2017/18",
    "2018/19",
    "2019/20",
    "2020/21",
    "2021/22",
    "2022/23",
    "2023/24",
    "2024/25"
]

pld_ptal_main = pld_ptal[
    pld_ptal["completion_financial_year"].isin(main_years)
].copy()

print("All PTAL-joined records:", len(pld_ptal))
print("Main analysis records:", len(pld_ptal_main))

print("All PTAL-joined units:", pld_ptal["completed_units"].sum())
print("Main analysis units:", pld_ptal_main["completed_units"].sum())

pld_ptal_main["completion_financial_year"].value_counts().sort_index()

All PTAL-joined records: 25852
Main analysis records: 25852
All PTAL-joined units: 225451.0
Main analysis units: 225451.0


completion_financial_year
2015/16    2746
2016/17    3940
2017/18    3288
2018/19    3249
2019/20    3179
2020/21    2368
2021/22    2494
2022/23     813
2023/24    2121
2024/25    1654
Name: count, dtype: int64

## 13. Generate the first batch of core summary tables

### 13.1. Overall PTAL distribution

In [45]:
analysis_df = pld_ptal_main.copy()
ptal_summary = (
    analysis_df
    .groupby("ptal_band", dropna=False)
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
)

ptal_summary["record_share"] = ptal_summary["records"] / ptal_summary["records"].sum()
ptal_summary["unit_share"] = ptal_summary["completed_units"] / ptal_summary["completed_units"].sum()

ptal_summary

,ptal_band,records,completed_units,record_share,unit_share
0,High,7261,89568.0,0.280868,0.397284
1,Low,9142,63459.0,0.353628,0.281476
2,Medium,9426,72077.0,0.364614,0.319701
3,NaN,23,347.0,0.000890,0.001539


In [46]:
ptal_summary.to_csv(
    OUTPUT_DIR / "summary_completed_units_by_ptal_band.csv",
    index=False
)

### 13.2 Annual × PTAL Distribution

In [47]:
annual_ptal_summary = (
    analysis_df
    .groupby(["completion_financial_year", "ptal_band"], dropna=False)
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
)

annual_ptal_summary.to_csv(
    OUTPUT_DIR / "summary_completed_units_by_year_ptal_band.csv",
    index=False
)

annual_ptal_summary.head()

,completion_financial_year,ptal_band,records,completed_units
0,2015/16,High,904,7418.0
1,2015/16,Low,839,6145.0
2,2015/16,Medium,1001,5875.0
3,2015/16,NaN,2,13.0
4,2016/17,High,1335,13295.0


### 13.3 Annual medium-high PTAL share

In [48]:
annual_base = (
    analysis_df
    .groupby("completion_financial_year")
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
)

annual_medium_high = (
    analysis_df[analysis_df["ptal_medium_high"]]
    .groupby("completion_financial_year")
    .agg(
        units_medium_high=("completed_units", "sum")
    )
    .reset_index()
)

annual_high = (
    analysis_df[analysis_df["ptal_high"]]
    .groupby("completion_financial_year")
    .agg(
        units_high=("completed_units", "sum")
    )
    .reset_index()
)

annual_accessibility = (
    annual_base
    .merge(annual_medium_high, on="completion_financial_year", how="left")
    .merge(annual_high, on="completion_financial_year", how="left")
)

annual_accessibility[["units_medium_high", "units_high"]] = (
    annual_accessibility[["units_medium_high", "units_high"]].fillna(0)
)

annual_accessibility["share_units_medium_high"] = (
    annual_accessibility["units_medium_high"] /
    annual_accessibility["completed_units"]
)

annual_accessibility["share_units_high"] = (
    annual_accessibility["units_high"] /
    annual_accessibility["completed_units"]
)

annual_accessibility.to_csv(
    OUTPUT_DIR / "summary_annual_accessibility_share.csv",
    index=False
)

annual_accessibility

,completion_financial_year,records,completed_units,units_medium_high,units_high,share_units_medium_high,share_units_high
0,2015/16,2746,19451.0,13293.0,7418.0,0.683410,0.381369
1,2016/17,3940,31612.0,23632.0,13295.0,0.747564,0.420568
2,2017/18,3288,26926.0,19367.0,9187.0,0.719268,0.341194
3,2018/19,3249,26280.0,18006.0,9481.0,0.685160,0.360769
4,2019/20,3179,27636.0,18181.0,10049.0,0.657874,0.363620
5,2020/21,2368,20728.0,13879.0,7783.0,0.669577,0.375482
6,2021/22,2494,21957.0,17151.0,9150.0,0.781118,0.416724
7,2022/23,813,11074.0,8328.0,3993.0,0.752032,0.360574
8,2023/24,2121,21476.0,17344.0,11012.0,0.807599,0.512758
9,2024/25,1654,18311.0,12464.0,8200.0,0.680684,0.447818


### 13.4 borough-level summary

In [49]:
# 13.4 Borough-level summary using spatially assigned boroughs

BOROUGH_COL = "borough_spatial"

borough_base = (
    analysis_df
    .groupby(BOROUGH_COL, dropna=False)
    .agg(
        records=("LPA Number", "count"),
        completed_units=("completed_units", "sum")
    )
    .reset_index()
)

borough_medium_high = (
    analysis_df[analysis_df["ptal_medium_high"]]
    .groupby(BOROUGH_COL, dropna=False)
    .agg(
        units_medium_high=("completed_units", "sum")
    )
    .reset_index()
)

borough_high = (
    analysis_df[analysis_df["ptal_high"]]
    .groupby(BOROUGH_COL, dropna=False)
    .agg(
        units_high=("completed_units", "sum")
    )
    .reset_index()
)

borough_summary = (
    borough_base
    .merge(borough_medium_high, on=BOROUGH_COL, how="left")
    .merge(borough_high, on=BOROUGH_COL, how="left")
)

borough_summary[["units_medium_high", "units_high"]] = (
    borough_summary[["units_medium_high", "units_high"]].fillna(0)
)

borough_summary["share_units_medium_high"] = (
    borough_summary["units_medium_high"] /
    borough_summary["completed_units"]
)

borough_summary["share_units_high"] = (
    borough_summary["units_high"] /
    borough_summary["completed_units"]
)

borough_summary = borough_summary.sort_values(
    "share_units_medium_high",
    ascending=False
)

borough_summary.to_csv(
    OUTPUT_DIR / "summary_borough_accessibility.csv",
    index=False
)

borough_summary.head(10)

,borough_spatial,records,completed_units,units_medium_high,units_high,share_units_medium_high,share_units_high
6,City of London,37,540.0,540.0,540.0,1.000000,1.000000
32,Westminster,837,7432.0,7400.0,6394.0,0.995694,0.860334
18,Islington,262,1880.0,1792.0,1592.0,0.953191,0.846809
21,Lambeth,954,7057.0,6474.0,4955.0,0.917387,0.702140
5,Camden,705,5563.0,5009.0,4112.0,0.900413,0.739170
27,Southwark,930,8759.0,7741.0,5331.0,0.883777,0.608631
10,Greenwich,307,3544.0,3075.0,1203.0,0.867664,0.339447
11,Hackney,1149,7491.0,6339.0,3613.0,0.846215,0.482312
3,Brent,1101,12664.0,10716.0,5762.0,0.846178,0.454991
31,Wandsworth,1174,12758.0,10704.0,6242.0,0.839003,0.489262


## 14. Save the final analysis data

In [50]:
pld_ptal.to_file(
    OUTPUT_DIR / "pld_completed_housing_with_ptal_all_years.gpkg",
    layer="pld_ptal_all",
    driver="GPKG"
)

pld_ptal_main.to_file(
    OUTPUT_DIR / "pld_completed_housing_with_ptal_main_2015_2025.gpkg",
    layer="pld_ptal_main",
    driver="GPKG"
)

pld_ptal_main.drop(columns="geometry").to_csv(
    OUTPUT_DIR / "pld_completed_housing_with_ptal_main_2015_2025_attributes.csv",
    index=False
)

In [51]:
# Main-period data coverage summary

main_years = [
    "2015/16", "2016/17", "2017/18", "2018/19", "2019/20",
    "2020/21", "2021/22", "2022/23", "2023/24", "2024/25"
]

core_main = pld_core[
    pld_core["completion_financial_year"].isin(main_years)
].copy()

valid_postcode_main = pld_clean[
    pld_clean["completion_financial_year"].isin(main_years)
].copy()

geocoded_main = pld_geocode[
    (pld_geocode["completion_financial_year"].isin(main_years)) &
    (pld_geocode["_merge"] == "both")
].copy()

not_geocoded_main = pld_geocode[
    (pld_geocode["completion_financial_year"].isin(main_years)) &
    (pld_geocode["_merge"] == "left_only")
].copy()

no_postcode_main_records = len(core_main) - len(valid_postcode_main)
no_postcode_main_units = (
    core_main["completed_units"].sum() -
    valid_postcode_main["completed_units"].sum()
)

main_coverage_summary = pd.DataFrame({
    "category": [
        "Core completed PLD records",
        "Excluded: no extractable postcode",
        "Excluded: postcode not matched to ONSPD",
        "Final geocoded sample"
    ],
    "records": [
        len(core_main),
        no_postcode_main_records,
        len(not_geocoded_main),
        len(geocoded_main)
    ],
    "completed_units": [
        core_main["completed_units"].sum(),
        no_postcode_main_units,
        not_geocoded_main["completed_units"].sum(),
        geocoded_main["completed_units"].sum()
    ]
})

main_coverage_summary["record_share_of_core"] = (
    main_coverage_summary["records"] /
    len(core_main)
)

main_coverage_summary["unit_share_of_core"] = (
    main_coverage_summary["completed_units"] /
    core_main["completed_units"].sum()
)

main_coverage_summary["record_share_of_core"] = main_coverage_summary["record_share_of_core"].map(lambda x: f"{x:.2%}")
main_coverage_summary["unit_share_of_core"] = main_coverage_summary["unit_share_of_core"].map(lambda x: f"{x:.2%}")

main_coverage_summary.to_csv(
    OUTPUT_DIR / "summary_main_analysis_coverage.csv",
    index=False
)

main_coverage_summary

,category,records,completed_units,record_share_of_core,unit_share_of_core
0,Core completed PLD records,27863,296746.0,100.00%,100.00%
1,Excluded: no extractable postcode,1941,70089.0,6.97%,23.62%
2,Excluded: postcode not matched to ONSPD,70,1206.0,0.25%,0.41%
3,Final geocoded sample,25852,225451.0,92.78%,75.97%


In [52]:
CORE_EXPORT_PATH = (
    OUTPUT_DIR /
    "pld_core_main_2015_2025.csv"
)

core_main.to_csv(
    CORE_EXPORT_PATH,
    index=False
)

print("Saved:")
print(CORE_EXPORT_PATH.resolve())
print("Core records:", len(core_main))
print(
    "Core completed units:",
    core_main["completed_units"].sum()
)

Saved:
C:\Users\YOLO\GitHub\CASA\dissertation\project\notebook\code\outputs\pld_core_main_2015_2025.csv
Core records: 27863
Core completed units: 296746.0
